# LangChain Components

**Source:** [LangChain Components | GenAI using LangChain | Video 2 | CampusX](https://www.youtube.com/watch?v=-xSJA8-o6Eg)

This video builds on Chapter 1 (introduction to LangChain / why it exists) and breaks down the **building blocks (components)** that make up every LangChain application. Think of these as LEGO pieces — you snap them together to build an LLM app instead of writing everything from scratch.

## Why components matter

Any LLM application, no matter how complex (a chatbot, a RAG system, an autonomous agent), is built from the *same* recurring set of pieces:

1. You need a **model** to talk to.
2. You need to **structure what you say to it** (prompts).
3. You need to **chain steps together** (chains).
4. You often need it to **remember past conversation** (memory).
5. You often need it to **use your own data** (indexes / document loaders / retrievers).
6. You sometimes need it to **take actions and use tools** (agents).

LangChain gives each of these a standard interface so you can swap providers (OpenAI ↔ Anthropic ↔ open-source) or swap strategies (in-memory ↔ vector DB) without rewriting your whole app.

## The 6 core components

### 1. Models
The interface to the actual AI model. LangChain standardizes three kinds:
- **LLMs** — take a string in, return a string out (older-style, text completion). e.g. `OpenAI()`
- **Chat Models** — take a list of messages (System/Human/AI) in, return a message out. This is what almost everything uses today. e.g. `ChatOpenAI()`
- **Embedding Models** — turn text into a vector of numbers (used for semantic search / RAG, not for generating text). e.g. `OpenAIEmbeddings()`

### 2. Prompts
The input you send to the model. Rarely just a raw string — LangChain gives tools to make prompts dynamic and reusable:
- **PromptTemplate** — a string with `{placeholders}` filled in at runtime.
- **ChatPromptTemplate** — same idea but for a list of chat messages (system/human/ai roles).
- **Few-shot prompting** — automatically inserting example Q&A pairs into the prompt to steer the model's style/format.
- Good prompt design is often the single biggest lever on output quality — cheaper and faster to iterate on than swapping models.

### 3. Chains
Chains connect components together so the output of one step becomes the input of the next (e.g. prompt → model → output parser).
- **LLMChain** — the classic single prompt → model call.
- **SequentialChain** — output of one chain feeds into the next.
- **Router chains** — pick which sub-chain to run based on the input.
- **Modern LangChain (v0.1+) mostly replaces the old `Chain` classes with LCEL — the `|` pipe operator**, e.g. `prompt | model | parser`. This is the direction the CampusX course moves toward in later videos.

### 4. Memory
LLM API calls are stateless by default — the model doesn't remember your last message unless you send the whole history back yourself. Memory components manage that history for you:
- **ConversationBufferMemory** — stores the raw conversation so far.
- **ConversationBufferWindowMemory** — keeps only the last *k* messages (controls cost/context length).
- **ConversationSummaryMemory** — periodically summarizes older history instead of keeping it verbatim (saves tokens on long chats).

### 5. Indexes (working with your own data / RAG)
This is the group of components that let an LLM answer questions about documents it was never trained on:
- **Document loaders** — pull in data from PDFs, websites, Notion, CSVs, etc.
- **Text splitters** — break large documents into small chunks (models have limited context windows).
- **Vector stores** — store chunk embeddings for fast similarity search (e.g. FAISS, Chroma, Pinecone).
- **Retrievers** — given a query, fetch the most relevant chunks from the vector store.
- Together these power **RAG (Retrieval-Augmented Generation)**: retrieve relevant chunks → stuff them into the prompt → ask the model to answer using only that context.

### 6. Agents
Instead of a fixed sequence of steps, an agent lets the **LLM itself decide** what to do next, in a loop:
1. LLM sees the goal + the tools it can call (search, calculator, code execution, custom APIs, ...).
2. LLM decides which tool to use and with what input.
3. The tool runs, its result is fed back to the LLM.
4. Repeat until the LLM decides it has the final answer.

This is the most powerful but least predictable component — useful for open-ended tasks, riskier for production because the model's reasoning can go off track.

## Quick mental model

```
Models        -> WHO you talk to
Prompts       -> WHAT you say to it
Chains / LCEL -> HOW multiple steps connect
Memory        -> WHAT it remembers
Indexes/RAG   -> WHAT extra data it can use
Agents        -> WHETHER it can decide + act on its own
```

A minimal LCEL example tying models + prompts + chains together (the pattern later videos in this course build on):

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")
parser = StrOutputParser()

chain = prompt | model | parser   # this pipe syntax IS a chain
print(chain.invoke({"topic": "LangChain components"}))
```

## Extra notes (beyond the video, but useful for this course)

- **LCEL (LangChain Expression Language)** — the modern `|` pipe way of composing components (`Runnable` objects). Newer CampusX videos and current LangChain docs favor this over the older `Chain` classes shown above; worth learning both since older tutorials/code still use `LLMChain` etc.
- **LangGraph** — LangChain's newer library for building more controllable, stateful agents as graphs instead of the simpler agent loop described above. Relevant once you go past basic agents.
- **LangSmith** — LangChain's official tool for tracing/debugging chains and agents (seeing exactly what prompt was sent, what the model returned, at each step). Very useful once apps get more than 1-2 steps.
- **Environment setup reminder** (from Chapter 1 / project setup): API keys go in a `.env` file and are loaded with `python-dotenv` (`load_dotenv()`), never hard-coded in the notebook.
- **Official docs** for going deeper on any component: https://python.langchain.com/docs/concepts/